# Lab 09 — QAOA: Intuición Guiada

El Quantum Approximate Optimization Algorithm (QAOA) resuelve problemas de optimización combinatoria mediante capas alternadas de operadores de coste y mezcla.

**Problema**: MaxCut en un grafo — particionar vértices en dos grupos maximizando el número de aristas entre grupos.

## 1. El problema MaxCut

Dado un grafo G=(V,E), MaxCut busca la partición S, V\S que maximice |{(u,v)∈E : u∈S, v∉S}|.

Formulación como Hamiltoniano Ising:
$$H_C = \frac{1}{2}\sum_{(i,j)\in E}(1 - Z_i Z_j)$$

Maximizar corte ≡ minimizar ⟨-H_C⟩.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

# Grafo de ejemplo: cuadrado (4 nodos, 4 aristas)
edges = [(0, 1), (1, 2), (2, 3), (3, 0)]
n_qubits = 4

# Hamiltoniano de coste MaxCut
pauli_list = []
for (i, j) in edges:
    # (1 - Zi*Zj) / 2 = 0.5*I - 0.5*ZiZj
    pauli_zz = ['I'] * n_qubits
    pauli_zz[i] = 'Z'
    pauli_zz[j] = 'Z'
    pauli_list.append((''.join(reversed(pauli_zz)), -0.5))  # negativo para minimizar

H_cost = SparsePauliOp.from_list(pauli_list)
print('Hamiltoniano de coste (MaxCut):')
print(H_cost)

# Corte máximo analítico para cuadrado = 4 aristas
print('\nCorte máximo esperado: 4 aristas (partición {0,2} vs {1,3})')

## 2. Circuito QAOA

El circuito QAOA con p capas alterna:
1. **Operador de coste**: e^{-iγH_C} = ∏_{(i,j)∈E} e^{-iγ(1-ZiZj)/2}
2. **Operador de mezcla**: e^{-iβH_B} = ∏_i e^{-iβXi} = ∏_i Rx(2β)

Inicialmente todos los qubits en |+⟩^n.

In [ ]:
def build_qaoa(edges: list, n_qubits: int, p: int) -> QuantumCircuit:
    """Circuito QAOA con p capas para MaxCut."""
    gammas = ParameterVector('γ', p)
    betas = ParameterVector('β', p)
    qc = QuantumCircuit(n_qubits)

    # Estado inicial |+⟩^n
    qc.h(range(n_qubits))

    for layer in range(p):
        # Operador de coste
        for (i, j) in edges:
            qc.cx(i, j)
            qc.rz(2 * gammas[layer], j)
            qc.cx(i, j)

        # Operador de mezcla
        for q in range(n_qubits):
            qc.rx(2 * betas[layer], q)

    return qc

qaoa_p1 = build_qaoa(edges, n_qubits, p=1)
print(qaoa_p1.draw('text'))
print(f'\nParámetros: {qaoa_p1.num_parameters}')

## 3. Optimización QAOA

Minimizamos ⟨H_cost⟩ respecto a los parámetros (γ, β). Para p=1 en el cuadrado, QAOA puede lograr un corte de ~3.5 (approx ratio ≈ 0.875).

In [ ]:
estimator = StatevectorEstimator()
ratio_history = []

def qaoa_cost(params: np.ndarray, qc: QuantumCircuit) -> float:
    job = estimator.run([(qc, H_cost, params)])
    val = float(job.result()[0].data.evs)
    ratio_history.append(-val / 4)  # ratio vs max=4
    return val

# p=1
qaoa = build_qaoa(edges, n_qubits, p=1)
np.random.seed(7)
x0 = np.random.uniform(0, np.pi, qaoa.num_parameters)
res = minimize(qaoa_cost, x0, args=(qaoa,), method='COBYLA',
               options={'maxiter': 300, 'rhobeg': 0.3})

print(f'Corte aproximado:  {-res.fun:.4f} aristas')
print(f'Corte máximo:      4.0000 aristas')
print(f'Ratio aproximación:{-res.fun/4:.4f}')

## 4. Distribución del estado solución

Medimos el estado óptimo para ver qué bitstrings dominan la distribución. El MaxCut del cuadrado tiene dos soluciones degeneradas: `0101` y `1010`.

In [ ]:
sampler = StatevectorSampler()

# Circuito con parámetros óptimos + medición
qaoa_meas = build_qaoa(edges, n_qubits, p=1)
qaoa_meas.measure_all()
job = sampler.run([(qaoa_meas, res.x)], shots=2048)
counts = job.result()[0].data.meas.get_counts()

# Ordenar por frecuencia
top_states = sorted(counts.items(), key=lambda x: -x[1])[:8]

fig, ax = plt.subplots(figsize=(8, 4))
bitstrings = [s for s, _ in top_states]
probs = [c / 2048 for _, c in top_states]
colors = ['green' if s in ('0101', '1010') else 'steelblue' for s in bitstrings]
ax.bar(bitstrings, probs, color=colors, alpha=0.85)
ax.set_xlabel('Bitstring', fontsize=12)
ax.set_ylabel('Probabilidad', fontsize=12)
ax.set_title('QAOA MaxCut — distribución de medidas (verde = solución óptima)', fontsize=12)
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Efecto de p en la calidad de la solución

A mayor número de capas p, QAOA puede aproximar mejor el óptimo. Para p→∞, QAOA converge al óptimo exacto (adiabatic theorem connection).

In [ ]:
print('Capas p | Corte aprox | Ratio')
print('-' * 35)
for p in [1, 2, 3]:
    ratio_history.clear()
    qaoa_p = build_qaoa(edges, n_qubits, p=p)
    x0_p = np.random.uniform(0, np.pi, qaoa_p.num_parameters)
    res_p = minimize(qaoa_cost, x0_p, args=(qaoa_p,),
                     method='COBYLA', options={'maxiter': 500})
    corte = -res_p.fun
    ratio = corte / 4
    print(f'  p={p}   | {corte:.4f}      | {ratio:.4f}')